In [31]:
import requests
import pandas as pd

In [25]:
import requests

url = "https://open.er-api.com/v6/latest/GBP"
response = requests.get(url)

print(response.status_code)

200


In [26]:
data = response.json()
data

{'result': 'success',
 'provider': 'https://www.exchangerate-api.com',
 'documentation': 'https://www.exchangerate-api.com/docs/free',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1770508951,
 'time_last_update_utc': 'Sun, 08 Feb 2026 00:02:31 +0000',
 'time_next_update_unix': 1770597101,
 'time_next_update_utc': 'Mon, 09 Feb 2026 00:31:41 +0000',
 'time_eol_unix': 0,
 'base_code': 'GBP',
 'rates': {'GBP': 1,
  'AED': 4.992383,
  'AFN': 88.796055,
  'ALL': 111.421842,
  'AMD': 513.516414,
  'ANG': 2.43332,
  'AOA': 1282.171658,
  'ARS': 1974.183489,
  'AUD': 1.944202,
  'AWG': 2.43332,
  'AZN': 2.309187,
  'BAM': 2.251273,
  'BBD': 2.718793,
  'BDT': 166.341594,
  'BGN': 2.206834,
  'BHD': 0.511133,
  'BIF': 4043.262136,
  'BMD': 1.359396,
  'BND': 1.729249,
  'BOB': 9.414836,
  'BRL': 7.136489,
  'BSD': 1.359396,
  'BTN': 123.043596,
  'BWP': 19.10053,
  'BYN': 3.879519,
  'BZD': 2.718793,
  'CAD': 1.857522,
  'CDF': 3107.880597,
  'CHF': 1.0556

In [27]:
gbp_to_inr = data["rates"]["INR"]
gbp_to_inr

123.044521

In [1]:
!pip install psycopg2-binary


   ---------------------------------------- 2.7/2.7 MB 19.6 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg2
import pandas as pd



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\prana\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\prana\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\prana\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\prana\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\prana\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\prana\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\prana\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\prana\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\prana\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\prana\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

In [15]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="postgres",   # default DB
    user="postgres",
    password="143Amresh@"
)


In [16]:
conn.autocommit = True


In [17]:
cursor = conn.cursor()


In [28]:
#cursor.execute("CREATE DATABASE pricing;")


In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS staging_exchange_rates (
    rate_date DATE PRIMARY KEY,
    gbp_to_inr NUMERIC
);
""")


In [20]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id TEXT PRIMARY KEY,
    title TEXT,
    price_gbp NUMERIC,
    price_inr NUMERIC,
    in_stock BOOLEAN,
    price_tier TEXT
);
""")


In [21]:
conn.commit()


In [29]:
cursor.execute("""
INSERT INTO staging_exchange_rates (rate_date, gbp_to_inr)
VALUES (CURRENT_DATE, %s)
ON CONFLICT (rate_date) DO NOTHING;
""", (gbp_to_inr,))

conn.commit()


In [39]:
df = pd.read_csv("books_raw.csv1")
df.head()

,title,price_gbp,availability,price_inr,in_stock,price_tier,product_id
0,A Light in the Attic,51.77,In stock,6370.014852,True,expensive,d682fd4bb45656063fc9eb088f20de3f
1,Tipping the Velvet,53.74,In stock,6612.412559,True,expensive,b771174bde49b2c24ef7ac178419435b
2,Soumission,50.10,In stock,6164.530502,True,expensive,0eeec5851efa6ebf8c0db9dfb6096e5e
3,Sharp Objects,47.82,In stock,5883.988994,True,expensive,86ff04a40b3e53b038b49a98207c0127
4,Sapiens: A Brief History of Humankind,54.23,In stock,6672.704374,True,expensive,c0e265b2cafdec9dfb4bc8c200c94d04


In [40]:
df["price_gbp"] = (
    df["price_gbp"]
    .astype(str)
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .astype(float)
)
df

,title,price_gbp,availability,price_inr,in_stock,price_tier,product_id
0,A Light in the Attic,51.77,In stock,6370.014852,True,expensive,d682fd4bb45656063fc9eb088f20de3f
1,Tipping the Velvet,53.74,In stock,6612.412559,True,expensive,b771174bde49b2c24ef7ac178419435b
2,Soumission,50.10,In stock,6164.530502,True,expensive,0eeec5851efa6ebf8c0db9dfb6096e5e
3,Sharp Objects,47.82,In stock,5883.988994,True,expensive,86ff04a40b3e53b038b49a98207c0127
4,Sapiens: A Brief History of Humankind,54.23,In stock,6672.704374,True,expensive,c0e265b2cafdec9dfb4bc8c200c94d04
5,The Requiem Red,22.65,In stock,2786.958401,True,expensive,92684102e1573edd50e5e8b1c8cbd79f
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4102.304330,True,expensive,c6c1ee76284bf714bc84f4c8c3cb4cb1
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,2206.188262,True,expensive,28e839563e49bfeac720f65881b7eb20
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,2780.806175,True,expensive,624397f941696d73072fdef59fe19b9e
9,The Black Maria,52.15,In stock,6416.771770,True,expensive,5720f7ba9b00df2a17e6f185ab3279e0


In [41]:
for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO products (
        product_id, title,
        price_gbp, price_inr,
        in_stock, price_tier
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    ON CONFLICT (product_id) DO UPDATE
    SET
        price_gbp = EXCLUDED.price_gbp,
        price_inr = EXCLUDED.price_inr,
        in_stock = EXCLUDED.in_stock,
        price_tier = EXCLUDED.price_tier;
    """, (
        row["product_id"],
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["in_stock"],
        row["price_tier"]
    ))


In [42]:
conn.commit()


In [43]:
cursor.execute("SELECT * FROM products LIMIT 5;")
cursor.fetchall()


[('d682fd4bb45656063fc9eb088f20de3f',
  'A Light in the Attic',
  Decimal('51.77'),
  Decimal('6370.01485217'),
  True,
  'expensive'),
 ('b771174bde49b2c24ef7ac178419435b',
  'Tipping the Velvet',
  Decimal('53.74'),
  Decimal('6612.412558540001'),
  True,
  'expensive'),
 ('0eeec5851efa6ebf8c0db9dfb6096e5e',
  'Soumission',
  Decimal('50.1'),
  Decimal('6164.5305021'),
  True,
  'expensive'),
 ('86ff04a40b3e53b038b49a98207c0127',
  'Sharp Objects',
  Decimal('47.82'),
  Decimal('5883.98899422'),
  True,
  'expensive'),
 ('c0e265b2cafdec9dfb4bc8c200c94d04',
  'Sapiens: A Brief History of Humankind',
  Decimal('54.23'),
  Decimal('6672.70437383'),
  True,
  'expensive')]